[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/03_analysis/C1_pipeline_analysis.ipynb)

# C1: Pipeline Analysis

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Calculate summary statistics** for housing pipeline data
2. **Write SQL queries** to analyze project distributions
3. **Compute conversion rates** (proposals → approvals → completions)
4. **Identify patterns** in large vs. small projects

## Why This Matters

Raw data tells you *what* is happening. Analysis tells you *why* it matters:
- Are projects getting approved? At what rate?
- How many units are stuck in review vs. under construction?
- Which project sizes have the highest success rates?

This notebook transforms permit records into policy insights.

## Key Metrics

- **Conversion rate**: % of proposed units that get built
- **Pipeline distribution**: Where projects are in the process
- **Concentration**: How much impact do top projects have?

---

## Overview

Analyze the housing development pipeline with SQL queries and summary statistics.

**Metrics:**
- Projects by status (count and units)
- Conversion rates (proposals -> approvals -> completions)
- Year-over-year trends

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config with resolved paths
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve relative paths to absolute
for key, value in CONFIG['paths'].items():
    if isinstance(value, str) and not value.startswith('http'):
        CONFIG['paths'][key] = str(ROOT / value)

print(f"✅ Project root: {ROOT}")
print(f"✅ Housing data: {CONFIG['paths']['housing_projects']}")

## 2. Load Data

In [ ]:
# Load from CSV
housing_path = Path(CONFIG['paths']['housing_projects'])
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    print(f"Total units: {df['net_units'].sum():,.0f}")

## 3. Projects by Status

In [ ]:
# Summary by status
if df is not None:
    status_summary = df.groupby('status').agg({
        'address_display': 'count',
        'net_units': 'sum'
    }).reset_index()
    
    status_summary.columns = ['Status', 'Projects', 'Units']
    status_summary = status_summary.sort_values('Units', ascending=False)
    
    print("Projects by Status:")
    print("="*60)
    display(status_summary)
    
    # Total
    print(f"\nTotal: {status_summary['Projects'].sum()} projects, {status_summary['Units'].sum():,.0f} units")

## 4. SQL Queries on Database

In [ ]:
# Query database directly
def query_db(sql):
    """Execute SQL query on housing database"""
    try:
        conn = sqlite3.connect(DB_PATH)
        result = pd.read_sql_query(sql, conn)
        conn.close()
        return result
    except Exception as e:
        print(f"Query error: {e}")
        return None

# Projects by size category
sql = """
SELECT 
    project_size_category,
    COUNT(*) as projects,
    SUM(net_units) as total_units,
    ROUND(AVG(net_units), 1) as avg_units
FROM projects
GROUP BY project_size_category
ORDER BY total_units DESC
"""

result = query_db(sql)
if result is not None:
    print("Projects by Size Category:")
    display(result)

In [ ]:
# Projects by year
sql = """
SELECT 
    CAST(year AS INTEGER) as year,
    COUNT(*) as projects,
    SUM(net_units) as total_units
FROM projects
WHERE year IS NOT NULL
GROUP BY year
ORDER BY year DESC
LIMIT 10
"""

result = query_db(sql)
if result is not None:
    print("Projects by Year:")
    display(result)

## 5. Conversion Rates Analysis

### Key Concept: The Pipeline Funnel

Not all proposed housing gets built. The "funnel" shows attrition:

```
PROPOSED (100%)
    ↓ Some withdrawn, denied, modified
APPROVED (60-80%)
    ↓ Some never start construction
UNDER CONSTRUCTION (40-60%)
    ↓ Some stall or are abandoned
COMPLETED (30-50%)
```

Understanding this funnel helps set realistic expectations for housing targets.

In [ ]:
# Pipeline conversion rates
if df is not None:
    # Define pipeline stages
    stages = {
        'Proposed/In Review': ['In Review', 'Under Review', 'Incomplete'],
        'Approved': ['Approved', 'Pending Final Action'],
        'Completed': ['Completed', 'Certificate']
    }
    
    print("Pipeline Conversion Analysis:")
    print("="*60)
    
    stage_counts = {}
    for stage_name, keywords in stages.items():
        mask = df['status'].str.contains('|'.join(keywords), case=False, na=False)
        count = mask.sum()
        units = df.loc[mask, 'net_units'].sum()
        stage_counts[stage_name] = {'projects': count, 'units': units}
        print(f"{stage_name}: {count} projects, {units:,.0f} units")
    
    # Calculate conversion rates
    total_projects = len(df)
    total_units = df['net_units'].sum()
    
    print(f"\nConversion Rates (by units):")
    if stage_counts.get('Proposed/In Review', {}).get('units', 0) > 0:
        review_to_approved = stage_counts.get('Approved', {}).get('units', 0) / total_units * 100
        print(f"  In Review -> Approved: {review_to_approved:.1f}%")

## 6. Large Projects Analysis

In [ ]:
# Top 20 largest projects
if df is not None:
    print("Top 20 Largest Projects:")
    print("="*60)
    
    top_20 = df.nlargest(20, 'net_units')[['address_display', 'net_units', 'status', 'year']]
    display(top_20)
    
    print(f"\nTop 20 represent: {top_20['net_units'].sum():,.0f} units ({top_20['net_units'].sum()/df['net_units'].sum()*100:.1f}% of total)")

## 7. Export Summary

In [ ]:
# Export analysis summary
if df is not None:
    summary_data = {
        'total_projects': len(df),
        'total_units': int(df['net_units'].sum()),
        'by_status': df.groupby('status')['net_units'].sum().to_dict(),
        'by_year': df.groupby('year')['net_units'].sum().to_dict()
    }
    
    summary_path = DATA_DIR / 'pipeline_analysis_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary_data, f, indent=2, default=str)
    
    print(f"Saved: {summary_path}")

---

## Summary

This notebook analyzed:
- Projects by status and size category
- Year-over-year trends
- Pipeline conversion rates
- Largest projects

**Next:** Run `C2_timeline_analysis.ipynb` for time-based analysis.